# Подготовка датасета



In [119]:
import pandas as pd
import numpy as np
import warnings
import re

warnings.filterwarnings('ignore')

df = pd.read_csv('2400_ekatalog.csv')
print(df.shape)
print(df.columns.tolist())

(2455, 13)
['name', 'min_price', 'max_price', 'headers', 'related_links', 'total_characteristics', 'char_Przekątna', 'char_Rozdzielczość', 'char_Matryca', 'char_Częstotliwość', 'char_Kontrast', 'char_Jasność', 'char_Złącza']


### Оценка нулов



In [120]:
missing = df.isnull().sum()
missing_pct = (missing / len(df)) * 100
missing_df = pd.DataFrame({
    'Missing Count': missing,
    'Percentage': missing_pct
})
missing_df[missing_df['Missing Count'] > 0]



,Missing Count,Percentage
min_price,1,0.040733
max_price,498,20.285132
headers,1,0.040733
related_links,39,1.588595
char_Przekątna,1,0.040733
char_Rozdzielczość,1,0.040733
char_Matryca,4,0.162933
char_Częstotliwość,29,1.181263
char_Kontrast,112,4.562118
char_Jasność,42,1.710794


### Выделение таргета

Таргет - среднее между минимальной и максимальной ценой. Нулы в комплементарных колонках(min/max) заполняются значением в присутствующей колонке.

In [121]:
df = df.copy()

min_missing = df['min_price'].isna() & df['max_price'].notna()
max_missing = df['max_price'].isna() & df['min_price'].notna()

df.loc[min_missing, 'min_price'] = df.loc[min_missing, 'max_price']
df.loc[max_missing, 'max_price'] = df.loc[max_missing, 'min_price']

df['target_price'] = ((df['min_price'] + df['max_price']) / 2) * 22.22 # для наглядности переводим злотые в рубли по курсу на момент парсинга

print(df.shape)
print(df['target_price'].describe())


(2455, 14)
count    2.454000e+03
mean     6.818177e+04
std      1.445101e+05
min      2.977480e+03
25%      1.765379e+04
50%      2.964703e+04
75%      6.112444e+04
max      2.836583e+06
Name: target_price, dtype: float64


### Обработка метрик



In [122]:
def extract_diagonal(value):
    try:
        return float(str(value).replace('"', '').strip())
    except:
        return np.nan

def extract_resolution(value):
    try:
        parts = str(value).split('(')[0].strip()
        width, height = parts.split('x')
        width, height = int(width), int(height)
        ratio = width / height if height > 0 else 0
        return width, height, ratio
    except:
        return np.nan, np.nan, np.nan

def extract_frequency(value):
    try:
        return int(str(value).replace('Hz', '').strip())
    except:
        return np.nan

def extract_contrast(value):
    value = str(value).replace(' ', '').strip()
    parts = str(value).split(':')[0]
    if parts == 'nan':
        parts = 1000

    return parts

def extract_brightness(value):
    try:
        return int(str(value).replace('cd/m²', '').strip())
    except:
        return np.nan

def extract_matrix_type(value):
    if pd.isna(value): 
        return 'Unknown'
    v = str(value).upper()
    for t in ['IPS', 'VA', 'TN', 'OLED']:
        if t in v: 
            return t
    return 'Other'

def extract_response_time(value):
    m = re.search(r'czas reakcji (\d+) ms', str(value))
    return int(m.group(1)) if m else np.nan



In [123]:
df['diagonal_inches'] = df['char_Przekątna'].apply(extract_diagonal)
df[['width_px', 'height_px', 'aspect_ratio']] = df['char_Rozdzielczość'].apply(
    lambda x: pd.Series(extract_resolution(x)))

df['PPI'] = np.sqrt(df['width_px'] ** 2 + df['height_px'] ** 2) / df['diagonal_inches']

df['frequency_hz'] = df['char_Częstotliwość'].apply(extract_frequency)
df['contrast_ratio'] = df['char_Kontrast'].apply(extract_contrast)
df['brightness_cd'] = df['char_Jasność'].apply(extract_brightness)
df['matrix_type'] = df['char_Matryca'].apply(extract_matrix_type)
df['response_time_ms'] = df['char_Matryca'].apply(extract_response_time)

df['total_pixels'] = (df['width_px'] * df['height_px']) / 1_000_000

print(f"Новые метрики: diagonal_inches, width_px, height_px, aspect_ratio, PPI, frequency_hz, contrast_ratio, brightness_cd, response_time_ms, total_pixels, matrix_type")


Новые метрики: diagonal_inches, width_px, height_px, aspect_ratio, PPI, frequency_hz, contrast_ratio, brightness_cd, response_time_ms, total_pixels, matrix_type


### Заполнение нулов для числовых метрик

In [124]:
def impute_numeric_by_group(df_to_impute, group_col, numeric_cols_to_impute):

    df_copy = df_to_impute.copy()

    for col in numeric_cols_to_impute:
        missing_before = df_copy[col].isnull().sum()
        if missing_before == 0:
            continue

        df_copy[col] = df_copy.groupby(group_col)[col].transform(lambda x: x.fillna(x.mean()))

        if df_copy[col].isnull().any():
            global_mean = df_copy[col].mean()
            df_copy[col].fillna(global_mean, inplace=True)


    return df_copy

numeric_features_to_impute = [
    'diagonal_inches', 'PPI', 'frequency_hz', 'contrast_ratio',
    'brightness_cd', 'response_time_ms', 'aspect_ratio'
]

df = impute_numeric_by_group(df, 'matrix_type', numeric_features_to_impute)

df[numeric_features_to_impute].isnull().sum()


diagonal_inches     0
PPI                 0
frequency_hz        0
contrast_ratio      0
brightness_cd       0
response_time_ms    0
aspect_ratio        0
dtype: int64

### Обработка категориальных метрик



In [125]:
def categorize_diagonal(diagonal):
    if pd.isna(diagonal): 
        return 'Unknown'
    if diagonal < 24:
        return 'Small'
    elif diagonal < 27:
        return 'Medium'
    elif diagonal < 32:
        return 'Large'
    else:
        return 'XLarge'

def categorize_resolution(width, height):
    if pd.isna(width) or pd.isna(height): 
        return 'Unknown'
    total_pixels = width * height
    if total_pixels >= 7680 * 4320:
        return '8K'
    elif total_pixels >= 3840 * 2160:
        return '4K'
    elif total_pixels >= 2560 * 1440:
        return 'QHD'
    elif total_pixels >= 1920 * 1080:
        return 'FullHD'
    else:
        return 'HD'

df['diagonal_category'] = df['diagonal_inches'].apply(categorize_diagonal)
df['resolution_category'] = df.apply(
    lambda r: categorize_resolution(r['width_px'], r['height_px']), axis=1)

df['is_gaming'] = (df['frequency_hz'] >= 120).astype(int)
df['is_premium'] = ((df['total_pixels'] >= 8) & (df['frequency_hz'] >= 120)).astype(int)

df['brand'] = df['headers']

df.columns

Index(['name', 'min_price', 'max_price', 'headers', 'related_links',
       'total_characteristics', 'char_Przekątna', 'char_Rozdzielczość',
       'char_Matryca', 'char_Częstotliwość', 'char_Kontrast', 'char_Jasność',
       'char_Złącza', 'target_price', 'diagonal_inches', 'width_px',
       'height_px', 'aspect_ratio', 'PPI', 'frequency_hz', 'contrast_ratio',
       'brightness_cd', 'matrix_type', 'response_time_ms', 'total_pixels',
       'diagonal_category', 'resolution_category', 'is_gaming', 'is_premium',
       'brand'],
      dtype='object')

### Удаление лишних колонок

In [126]:
df = df.drop(columns=['total_characteristics', 'char_Przekątna', 'related_links', 'char_Rozdzielczość', 'char_Matryca', 'char_Częstotliwość', 'char_Kontrast', 'char_Jasność', 'char_Złącza', 'headers']).copy()

### Сохранение результата



In [127]:
df.to_csv('cleaned_data.csv', index=False)
df.shape

(2455, 20)